# Arm A — Global (Non-Stratified) Baseline

This notebook implements **Arm A** from the proposal (§3.3): a single classifier trained on the
**full patient population without stratification**. It serves as the global baseline against which
the clinically-defined subgroups (Arm B) and data-driven clusters (Arm C) are compared.

It follows the evaluation protocol in §3.5 and the preprocessing rules in §3.2:

- Two classifiers: **logistic regression** (interpretable linear) and **random forest** (non-linear comparator).
- **Stratified k-fold cross-validation** using **fixed fold partitions that are saved and reused by every arm**.
- Metrics: **accuracy, F1-score, ROC-AUC**, reported as **mean ± standard deviation across folds**.
- All parameter-estimating preprocessing (scaling, encoding) and hyperparameter selection are fit
  **on the training folds only**, to prevent data leakage.
- **Out-of-fold predictions are saved** so Arms B and C can be compared to this baseline on identical folds.


## 1. init setup

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# confg:
RANDOM_STATE = 42                     
K_OUTER      = 5 
K_INNER      = 5

## 2. Load data define X / y

**leakage

In [8]:
import pandas as pd
df = pd.read_csv("heart+disease/cleveland_clean.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,check
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,False
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,True
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,True
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,False
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,False


num represents the severity of heart disease. and check is representing whether a patient has heart dsease. So we need to remove them, oterwise there will be data leakage

In [9]:
y = df["check"].astype(int)            # target: 0 = no disease, 1 = disease
X = df.drop(columns=["num", "check"])

print("Samples:", len(df), "  Features:", X.shape[1])
print("Class balance:")
print(y.value_counts().rename({0: "no disease", 1: "disease"}))
print("Positive rate: {:.3f}".format(y.mean()))

Samples: 297   Features: 13
Class balance:
check
no disease    160
disease       137
Name: count, dtype: int64
Positive rate: 0.461


## 3. Feature groups and preprocessing

Feature types are handled per §3.2. Preprocessing lives **inside a Pipeline** so it is re-fit on the
training portion of each fold, -- no statistics leak from validation data.

- **Continuous** (`age, trestbps, chol, thalach, oldpeak`) -> standardised (`StandardScaler`).
- **Nominal** (`cp, restecg, slope, thal`) -> these are category codes, not magnitudes.
- **Binary / count** (`sex, fbs, exang, ca`) -> passed through unchanged (already 0/1, or a 0–3 count).

In [10]:
continuous  = ["age", "trestbps", "chol", "thalach", "oldpeak"]
nominal     = ["cp", "restecg", "slope", "thal"]
passthrough = ["sex", "fbs", "exang", "ca"] # we don't need to do anything

preprocess = ColumnTransformer([
    ("num",  StandardScaler(),                        continuous),
    ("cat",  OneHotEncoder(handle_unknown="ignore"),  nominal),
    ("pass", "passthrough",                           passthrough),
])

## 4. cross-validation folds

The proposal requires **the same fold partitions across all arms** (§3.5). We assign every patient a
fold id **once**, stratified by the target, and **save it to `fold_id.csv`**. Arms B and C load this
same file, so a patient always sits in the same validation fold regardless of arm. 

In [11]:
skf = StratifiedKFold(n_splits=K_OUTER, shuffle=True, random_state=RANDOM_STATE)

fold_id = np.empty(len(df), dtype=int)
for k, (_, val_idx) in enumerate(skf.split(X, y)):
    fold_id[val_idx] = k

pd.Series(fold_id, name="fold").to_csv("fold_id.csv", index=False)
print("Fold sizes:", np.bincount(fold_id))
print("Positives per fold:", np.bincount(fold_id[y.values == 1]))

Fold sizes: [60 60 59 59 59]
Positives per fold: [28 28 27 27 27]


## 5. Models and hyperparameter selection

Two classifiers, each with a **small** hyperparameter grid (§3.4: "no extensive optimisation").
The **same procedure is applied in every arm**: hyperparameters are chosen by an **inner
cross-validation** on the training folds only (nested CV), so selection never sees the outer
validation data. 

this code block should be the same across different arms

In [12]:
models = {
    "logreg": (
        Pipeline([("pre", preprocess),
                  ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
        {"clf__C": [0.01, 0.1, 1, 10]},
    ),
    "rf": (
        Pipeline([("pre", preprocess),
                  ("clf", RandomForestClassifier(random_state=RANDOM_STATE))]),
        {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 5, 10]},
    ),
}

## 6. Cross-validation

For each model and each outer fold: select hyperparameters by inner CV on the training data, refit on
the full training portion, then predict on the held-out fold. We record accuracy, F1, and ROC-AUC per
fold and store the **out-of-fold predicted probabilities** for every patient.

In [13]:
results_rows = []
oof = {"fold": fold_id, "y": y.values}

for name, (pipe, grid) in models.items():
    accs, f1s, aucs, chosen = [], [], [], []
    oof_proba = np.zeros(len(df))

    for k in range(K_OUTER):
        train, validation = (fold_id != k), (fold_id == k)
        inner = StratifiedKFold(n_splits=K_INNER, shuffle=True, random_state=RANDOM_STATE)
        search = GridSearchCV(pipe, grid, cv=inner, scoring="roc_auc", n_jobs=-1)
        search.fit(X[train], y[train])

        best = search.best_estimator_
        proba = best.predict_proba(X[validation])[:, 1] # take out the ppl that actually has heart disease
        pred  = (proba >= 0.5).astype(int)


        oof_proba[validation] = proba
        accs.append(accuracy_score(y[validation], pred))
        f1s.append(f1_score(y[validation], pred))
        aucs.append(roc_auc_score(y[validation], proba))
        chosen.append(search.best_params_)

    oof[f"proba_{name}"] = oof_proba
    results_rows.append({
        "model": name,
        "accuracy_mean": np.mean(accs), "accuracy_std": np.std(accs),
        "f1_mean":       np.mean(f1s),  "f1_std":       np.std(f1s),
        "rocauc_mean":   np.mean(aucs), "rocauc_std":   np.std(aucs),
    })
    print(f"{name:7s} | ACC {np.mean(accs):.3f} +/- {np.std(accs):.3f}"
          f" | F1 {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}"
          f" | AUC {np.mean(aucs):.3f} +/- {np.std(aucs):.3f}")
    print("         chosen params per fold:", chosen)

logreg  | ACC 0.842 +/- 0.056 | F1 0.820 +/- 0.058 | AUC 0.902 +/- 0.040
         chosen params per fold: [{'clf__C': 0.1}, {'clf__C': 1}, {'clf__C': 1}, {'clf__C': 0.1}, {'clf__C': 0.1}]
rf      | ACC 0.815 +/- 0.058 | F1 0.789 +/- 0.068 | AUC 0.899 +/- 0.051
         chosen params per fold: [{'clf__max_depth': 5, 'clf__n_estimators': 200}, {'clf__max_depth': 5, 'clf__n_estimators': 200}, {'clf__max_depth': 10, 'clf__n_estimators': 200}, {'clf__max_depth': 10, 'clf__n_estimators': 200}, {'clf__max_depth': 5, 'clf__n_estimators': 400}]


## 7. Results summary

In [15]:
results = pd.DataFrame(results_rows)
results_display = results.set_index("model").round(3)
results_display

,accuracy_mean,accuracy_std,f1_mean,f1_std,rocauc_mean,rocauc_std
model,,,,,,
logreg,0.842,0.056,0.820,0.058,0.902,0.040
rf,0.815,0.058,0.789,0.068,0.899,0.051


## 8. Save outputs to other files

Two files are written for downstream use:

- **`armA_results.csv`** — mean ± std of each metric (goes into your results table).
- **`armA_oof_predictions.csv`** — per-patient fold id, true label, and predicted probability for each
  model. Arms B and C produce the same file on the **same folds**; comparing these three files answers
  RQ1–RQ3.

In [17]:
results.to_csv("armA_results.csv", index=False)
pd.DataFrame(oof).to_csv("armA_predictions.csv", index=False)
print("Saved fold_id.csv, armA_results.csv, armA_predictions.csv")

Saved fold_id.csv, armA_results.csv, armA_predictions.csv
